## working with agents on top of carteirinha extracted database

### this also should incorporate the base workflow style:
input: blob, id -> image, id -> llm ->  output: convenio, plano, nome da pessoa e número da carteirinha


In [33]:
import os
import base64
import sys
from dotenv import load_dotenv
from typing import Any, Dict, List, Optional, Tuple
from PIL import Image   # noqa: F401
from pydantic import BaseModel as PydanticBaseModel
from pydantic import Field
from pydantic_settings import BaseSettings
import oracledb
import io
import cv2
import fitz
import numpy as np
import boto3
from botocore.config import Config
import json
import re
import time
from tqdm import tqdm
import mariadb
import pandas as pd
from collections import defaultdict
from dataclasses import dataclass, field
#strands agetinc workflow

#dont limit the visualization of all the colluns of a pandas dataframe
pd.set_option("display.max_columns", None)

# Add project root to Python path so we can import from app module
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)
# Now we can import from app (after adding to sys.path)
from app.utils.logger import get_logger
# Load environment variables from the project root directory
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)

logger = get_logger(name=__name__)

## credentials config

In [34]:
@dataclass
class AppConstants:
    BEDROCK_DEFAULT_MODEL_ID: str = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
    BEDROCK_DEFAULT_MODEL_VERSION: str = "bedrock-2023-05-31"
    DEFAULT_PROMPTS_DIR: str = "prompts/"
    S3_BUCKET_NAME: str = "agente-ai-carteirinha"
    S3_RESULTS_PREFIX: str = "resultados"
    S3_DEBUG_PREFIX: str = "debug"
    STREAMING: bool = False
    CACHE_PROMPT = "default"
    RETRIES: Dict[str, int] = field(default_factory=lambda: {"max_attempts": 3, "mode": "standard"})
    CONNECTION_TIMEOUT: int = 5
    READ_TIMEOUT: int = 60
    TEMPERATURE: float = 0.05
    TOP_P: float = 0.95

In [35]:
class Settings(BaseSettings):
    """Carrega e valida as configurações a partir de variáveis de ambiente."""

    ORACLE_USER: str
    ORACLE_PASSWORD: str
    ORACLE_DSN: str
    ORACLE_INSTANT_CLIENT_PATH: Optional[str] = Field(
        None, alias="oracle_instant_client_path"
    )
    AWS_ACCESS_KEY_ID: str
    AWS_SECRET_ACCESS_KEY: str
    AWS_BEDROCK_REGION: str
    BEDROCK_MODEL_ID: str = AppConstants.BEDROCK_DEFAULT_MODEL_ID
    BEDROCK_MODEL_VERSION: str = AppConstants.BEDROCK_DEFAULT_MODEL_VERSION
    AWS_SERVICE_NAME: str
    MARIADB_USER: str
    MARIADB_PASSWORD: str
    MARIADB_HOST: str
    MARIADB_PORT: int = 3306
    MARIADB_DATABASE: str
    API_BASE_URL: Optional[str] = Field(None, alias="api_base_url")
    API_USERNAME: Optional[str] = Field(None, alias="username")
    API_PASSWORD: Optional[str] = Field(None, alias="password")

    class Config:
        env_file = ".env"
        env_file_encoding = "utf-8"

In [36]:
def criar_boto3_client(
    service_name: str, settings: Settings, config: Optional[Config] = None
) -> boto3.client:
    try:
        logger.info(
            f"Criando cliente {service_name.upper()} para a região: {settings.AWS_BEDROCK_REGION}..."
        )
        client = boto3.client(
            service_name,
            region_name=settings.AWS_BEDROCK_REGION,
            aws_access_key_id=settings.AWS_ACCESS_KEY_ID,
            aws_secret_access_key=settings.AWS_SECRET_ACCESS_KEY,
            config=config,
        )
        logger.info(f"Cliente {service_name.upper()} criado com sucesso.")
        return client
    except Exception as e:
        logger.critical(f"Não foi possível criar o cliente {service_name.upper()}: {e}")
        raise


## geting the service ready to use
### model configuration

In [37]:
easy_prompt = "what are llm models?"

In [38]:
def _extract_json_from_response(self, raw_text: str) -> str:
    """
    Extrai JSON de diferentes formatos de resposta do LLM.
    Tenta múltiplos métodos de extração para maximizar compatibilidade.
    """
    # Method 1: JSON em blocos de código (```json ... ```)
    code_block_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw_text, re.DOTALL)
    if code_block_match:
        logger.info("JSON extraído de bloco de código")
        return code_block_match.group(1).strip()
    
    # Method 2: JSON standalone (sem blocos de código)
    json_match = re.search(r'\{.*\}', raw_text, re.DOTALL)
    if json_match:
        logger.info("JSON extraído diretamente do texto")
        return json_match.group(0).strip()

def _validate_and_clean_json(self, json_str: str) -> str:
    """
    Valida e limpa a string JSON antes da validação Pydantic.
    """
    try:
        # Tenta fazer parse para verificar se é JSON válido
        parsed = json.loads(json_str)
        # Se chegou até aqui, o JSON é válido
        return json_str
    except json.JSONDecodeError as e:
        logger.warning(f"JSON inválido detectado: {e}")
        # Tenta algumas correções comuns
        cleaned = json_str.strip()
        # Remove possíveis caracteres extras no início/fim
        cleaned = re.sub(r'^[^\{]*', '', cleaned)
        cleaned = re.sub(r'[^\}]*$', '', cleaned)
        try:
            json.loads(cleaned)
            return cleaned
        except json.JSONDecodeError:
            raise ValueError(f"Não foi possível corrigir o JSON: {e}")

In [39]:
system_prompt = fr"""
Você é um especialista em extração estruturada de dados.  
Sua tarefa: analisar o texto e retornar **apenas** as informações de carteirinhas de convênio de saúde em JSON válido.  
⚠️ Retorne **somente JSON**, sem explicações, comentários ou texto adicional.

Campos a extrair:
- "convenio": Nome do convênio
- "plano": Tipo do plano (ex.: "Plano Prata")
- "nome_pessoa": Nome completo do beneficiário
- "numero_carteirinha": Número da carteirinha processado conforme regras abaixo

Regras de limpeza:
1. Remover caracteres especiais usando regex `[^a-zA-Z0-9\s]`
2. Remover espaços no início e fim
3. Aplicar regras especiais por convênio antes de validar tamanho
4. Validar tamanho conforme lista de mapeamento
5. Se algum campo não puder ser identificado ou validado, retornar null

Regras especiais por convênio:
- CEMIG SAUDE: Se houver dois números, use a matrícula do beneficiário (não a matrícula antiga)
- SUL AMERICA: Se o número tiver mais de 17 dígitos, remover os 3 primeiros dígitos e manter os 17 últimos.
- Outros convênios: Validar tamanho conforme tabela; se não estiver na lista, retornar null

Tabela de mapeamento (convênio, número de dígitos esperado):
[
("STELLANTIS SAUDE MG", 17),
("SUL AMERICA", "variavel"),
("CASSI", 16),
("CAIXA ECONOMICA FEDERAL", 11),
("BLUE COMPANY", 16),
("POSTAL SAUDE - CORREIOS", 16),
("IPSM", 16),
("UNIMED SEGUROS", 16),
("BRADESCO", 15),
("BRADESCO OPERADORA", 15),
("PLAN ASSISTE - MPF", 14),
("CARE PLUS", 12),
("PETROBRAS - REGAP", 12),
("VALE - AMS", 12),
("FUNDAFFEMG", 12),
("CEMIG SAUDE", "variavel"),
("VALE - PASA", 10),
("AMIL", 9),
("AMIL VM (ANTIGA GOLDEN CROSS)", 9),
("COPASS", 8),
("SPA SAUDE", 5)
]

Exemplo:
Texto: "Paciente João da Silva, convênio SUL AMERICA, carteirinha 12345678901234567890"
JSON esperado:
{{
  "convenio": "SUL AMERICA",
  "plano": null,
  "nome_pessoa": "João da Silva",
  "numero_carteirinha": "45678901234567890"
}}

"""

In [40]:
# user input 
# working with AWS texttract to get text from pdf

path_txt = "/home/joao/projects/company_projects/carteirinha-api/POC/keyValues.csv"

#transform this csv file into a txt file

import pandas as pd

# Load CSV
df = pd.read_csv(path_txt)

# Save as TXT (tab-separated)
df.to_csv("/home/joao/projects/company_projects/carteirinha-api/POC/rawText.txt", sep="\t", index=False)

# Read the text file
with open("/home/joao/projects/company_projects/carteirinha-api/POC/rawText.txt", "r") as file:
    raw_text = file.read()

In [41]:


# Final payload
body = {
    "anthropic_version": app_constants.BEDROCK_DEFAULT_MODEL_VERSION,
    "max_tokens": 4096,
    ##### using thinking ##########
    "temperature": 1,
    "thinking": {
        "type": "enabled",
        "budget_tokens": 2048
    },
    ##############################
    "messages": [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [{"type": "text", "text": raw_text}]}
    ]
}


In [42]:

app_constants = AppConstants()
settings = Settings()

# Create a custom boto3 session
bedrock_client = criar_boto3_client("bedrock-runtime", settings)
model_id = settings.BEDROCK_MODEL_ID
model_version = settings.BEDROCK_MODEL_VERSION

# Create a Bedrock model with the custom session


{"timestamp": "2025-08-13T13:18:19", "level": "INFO", "name": "__main__", "message": "Criando cliente BEDROCK-RUNTIME para a região: us-east-1...", "filename": "788707453.py", "lineno": 5}
{"timestamp": "2025-08-13T13:18:19", "level": "INFO", "name": "__main__", "message": "Cliente BEDROCK-RUNTIME criado com sucesso.", "filename": "788707453.py", "lineno": 15}


In [44]:
response = bedrock_client.invoke_model(
    body=body,
    modelId=model_id)

ParamValidationError: Parameter validation failed:
Invalid type for parameter body, value: {'anthropic_version': 'bedrock-2023-05-31', 'max_tokens': 4096, 'temperature': 1, 'thinking': {'type': 'enabled', 'budget_tokens': 2048}, 'messages': [{'role': 'system', 'content': [{'type': 'text', 'text': '\nVocê é um especialista em extração estruturada de dados.  \nSua tarefa: analisar o texto e retornar **apenas** as informações de carteirinhas de convênio de saúde em JSON válido.  \n⚠️ Retorne **somente JSON**, sem explicações, comentários ou texto adicional.\n\nCampos a extrair:\n- "convenio": Nome do convênio\n- "plano": Tipo do plano (ex.: "Plano Prata")\n- "nome_pessoa": Nome completo do beneficiário\n- "numero_carteirinha": Número da carteirinha processado conforme regras abaixo\n\nRegras de limpeza:\n1. Remover caracteres especiais usando regex `[^a-zA-Z0-9\\s]`\n2. Remover espaços no início e fim\n3. Aplicar regras especiais por convênio antes de validar tamanho\n4. Validar tamanho c